# S09 · Обвинение детектора

6 ноября 2026. Выполняйте ячейки сверху вниз с чистого ядра. Полный пример с результатами.

## C01 · Читаем тексты

Прочитайте data/coat-texts.csv: настоящий русский корпус текстов людей и нейросетей. ai=1: написала нейросеть, ai=0: человек. Четыре числа о тексте уже посчитаны: длина слова, доля разных слов, личные слова (я, мы, мне...) на 100 слов, запятые на 100 слов. [Данные и отбор](../../data/COAT-README.md).

In [1]:
import csv
from pathlib import Path

import numpy as np
from sklearn.linear_model import LogisticRegression

folder = Path.cwd()
if folder.name == "solutions":
    folder = folder.parent
if folder.name == "notebooks":
    folder = folder.parent
if folder.name != "starter":
    folder = folder / "starter"
with (folder / "data/coat-texts.csv").open(encoding="utf-8") as file:
    rows = list(csv.DictReader(file))
print("Текстов:", len(rows))
print(next(row["text"] for row in rows if "старостой" in row["text"]))


Текстов: 10000
моя подруга: я стала старостой и мне за это прибавят три балла в рейтинге!! я: мой универ: я: мой универ: ну и шо ты смотришь на меня


## C02 · Обучаем детектор

Соберите таблицу из четырёх чисел. Обучите детектор на train и посчитайте долю верных ответов на test при правиле: оценка не меньше 0.5 значит «нейросеть». В обеих частях половина текстов от нейросети.

In [2]:
names = ["word_length", "unique_share", "personal_per_100", "commas_per_100"]
X = np.array([[float(row[name]) for name in names] for row in rows])
y = np.array([int(row["ai"]) for row in rows])
part = np.array([row["split"] for row in rows])
train, test = part == "train", part == "test"
model = LogisticRegression(max_iter=2000)
model.fit(X[train], y[train])
scores = model.predict_proba(X)[:, 1]
print(dict(zip(names, model.coef_[0].round(2).tolist())))
print("Доля верных на test:", ((scores[test] >= 0.5) == y[test]).mean())


{'word_length': -0.15, 'unique_share': -0.37, 'personal_per_100': 0.13, 'commas_per_100': -0.1}
Доля верных на test: 0.663


## C03 · Две ошибки детектора

Precision и recall вы считали в S4. Здесь нужны две доли на 1000 текстов нейросети и 1000 текстов людей из test: какую долю нейротекстов детектор находит и какую долю людей обвиняет.

In [3]:
humans = test & (y == 0)
machines = test & (y == 1)
caught = (scores[machines] >= 0.5).mean()
false_alarm = (scores[humans] >= 0.5).mean()
print("Находит нейротекстов:", caught)
print("Обвиняет людей:", false_alarm)


Находит нейротекстов: 0.65
Обвиняет людей: 0.324


## C04 · Сколько обвинений верны

Запишите расчёт с доски функцией: верные обвинения делим на все обвинения. Проверьте рекламные 98% и 2%, затем наш детектор при той же доле списывающих 5%.

In [4]:
def share_guilty(base, caught, false_alarm):
    guilty = base * caught
    honest = (1 - base) * false_alarm
    return guilty / (guilty + honest)

print("Реклама, 5%:", round(share_guilty(0.05, 0.98, 0.02), 3))
print("Реклама, 1%:", round(share_guilty(0.01, 0.98, 0.02), 3))
print("Наш детектор, 5%:", round(share_guilty(0.05, caught, false_alarm), 3))


Реклама, 5%: 0.721
Реклама, 1%: 0.331
Наш детектор, 5%: 0.096


## C05 · Вы декан

На 1000 студентов, из которых 50 списали, посчитайте для порогов 0.5, 0.7 и 0.9: сколько списавших поймано и сколько честных обвинено. Доли берём из test.

In [5]:
for threshold in [0.5, 0.7, 0.9]:
    found = 50 * (scores[machines] >= threshold).mean()
    wrong = 950 * (scores[humans] >= threshold).mean()
    print(threshold, "поймано:", round(found, 1), "честных обвинено:", round(wrong, 1))


0.5 поймано: 32.5 честных обвинено: 307.8
0.7 поймано: 14.4 честных обвинено: 39.9
0.9 поймано: 2.5 честных обвинено: 3.8


## C06 · Кого обвиняют чаще

Разделите тексты людей из test на две группы: есть личные слова (я, мы, мне...) или их нет. Какую долю каждой группы детектор обвиняет при пороге 0.5?

In [6]:
personal = X[:, 2] > 0
print("С личными словами:", humans[personal].sum(), (scores[humans & personal] >= 0.5).mean())
print("Без личных слов:", humans[~personal].sum(), (scores[humans & ~personal] >= 0.5).mean())


С личными словами: 186 0.6666666666666666
Без личных слов: 814 0.2457002457002457
